# Demo 09: Linux CUDA LoRA fine-tuning for IT-support ticket classification

Train a BF16 LoRA adapter on one CUDA NVIDIA GPU. This notebook loads only training and validation records; it never loads the held-out test split.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, set_seed
from trl import SFTConfig, SFTTrainer


def find_project_root(start_directory: Path) -> Path:
    """Return the repository root containing the project instructions."""
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Could not find the project root. Start JupyterLab from the repository root.')


def require_cuda_bfloat16() -> torch.device:
    """Require a CUDA GPU with BF16 support and return cuda:0."""
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. Install CUDA-enabled PyTorch and verify the NVIDIA driver before running this notebook.')
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError('The selected CUDA GPU does not support BF16. This notebook requires a BF16-capable NVIDIA GPU.')
    return torch.device('cuda:0')


def to_prompt_completion(record: dict) -> dict:
    """Convert a ticket record to TRL prompt/completion format."""
    return {'prompt': [record['messages'][0]], 'completion': [record['messages'][1]]}


def validate_split(split_name: str, dataset) -> set[str]:
    """Validate ticket records and return ticket texts without displaying them."""
    ticket_texts = set()
    for index, record in enumerate(dataset):
        messages = record.get('messages')
        if not isinstance(messages, list) or len(messages) != 2:
            raise ValueError(f'{split_name} record {index} must contain exactly two messages.')
        if [message.get('role') for message in messages] != ['user', 'assistant']:
            raise ValueError(f'{split_name} record {index} must have user and assistant roles.')
        ticket_text = messages[0].get('content', '').strip()
        if not ticket_text or ticket_text in ticket_texts:
            raise ValueError(f'{split_name} record {index} has empty or duplicate ticket text.')
        ticket_texts.add(ticket_text)
        try:
            output = json.loads(messages[1].get('content', ''))
        except json.JSONDecodeError as error:
            raise ValueError(f'{split_name} record {index} has invalid assistant JSON.') from error
        if not isinstance(output, dict) or set(output) != {'category', 'severity', 'summary'}:
            raise ValueError(f'{split_name} record {index} must contain category, severity, and summary only.')
        if output['category'] not in ALLOWED_CATEGORIES or output['severity'] not in ALLOWED_SEVERITIES:
            raise ValueError(f'{split_name} record {index} has an unsupported label.')
        if not isinstance(output['summary'], str) or not output['summary'].strip():
            raise ValueError(f'{split_name} record {index} has an empty summary.')
    return ticket_texts


def calculate_validation_generation_metrics(model, tokenizer, validation_dataset) -> dict[str, float]:
    """Generate validation responses and return aggregate task metrics."""
    counts = {'valid': 0, 'category': 0, 'severity': 0, 'joint': 0}
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    model.eval()
    try:
        for start in range(0, len(validation_dataset), VALIDATION_GENERATION_BATCH_SIZE):
            records = validation_dataset.select(range(start, min(start + VALIDATION_GENERATION_BATCH_SIZE, len(validation_dataset))))
            prompts = [tokenizer.apply_chat_template([record['messages'][0]], tokenize=False, add_generation_prompt=True, enable_thinking=False) for record in records]
            inputs = {name: value.to(DEVICE) for name, value in tokenizer(prompts, return_tensors='pt', padding=True).items()}
            with torch.inference_mode():
                output_ids = model.generate(**inputs, max_new_tokens=GENERATION_MAX_NEW_TOKENS, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
            predictions = tokenizer.batch_decode(output_ids[:, inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
            for record, prediction in zip(records, predictions):
                try:
                    generated = json.loads(prediction.strip())
                    valid = isinstance(generated, dict) and set(generated) == {'category', 'severity', 'summary'}
                except json.JSONDecodeError:
                    generated, valid = {}, False
                reference = json.loads(record['messages'][1]['content'])
                category = valid and generated.get('category') == reference['category']
                severity = valid and generated.get('severity') == reference['severity']
                counts['valid'] += valid; counts['category'] += category; counts['severity'] += severity; counts['joint'] += category and severity
    finally:
        tokenizer.padding_side = original_padding_side
    total = len(validation_dataset)
    return {'category_accuracy': counts['category'] / total, 'severity_accuracy': counts['severity'] / total, 'joint_accuracy': counts['joint'] / total, 'valid_json_rate': counts['valid'] / total}


class ValidationGenerationMetricsCallback(TrainerCallback):
    """Add deterministic task metrics after each validation epoch."""
    def __init__(self, model, tokenizer, validation_dataset):
        self.model, self.tokenizer, self.validation_dataset = model, tokenizer, validation_dataset
        self.epoch_metrics = []

    def on_evaluate(self, args, state, control, **kwargs):
        metrics = kwargs.get('metrics', {})
        generated = calculate_validation_generation_metrics(self.model, self.tokenizer, self.validation_dataset)
        training_loss = next((entry['loss'] for entry in reversed(state.log_history) if 'loss' in entry), float('nan'))
        row = {'epoch': state.epoch, 'training_loss': training_loss, 'validation_loss': metrics['eval_loss'], **generated}
        metrics['eval_joint_accuracy'] = generated['joint_accuracy']
        self.epoch_metrics.append(row)
        display(Markdown(f"Epoch {row['epoch']:.0f}: category {row['category_accuracy']:.1%}, severity {row['severity_accuracy']:.1%}, joint {row['joint_accuracy']:.1%}, valid JSON {row['valid_json_rate']:.1%}"))
        return control


## Editable local paths

**Change these three settings for the Linux host before running the notebook.** `DATASET_DIRECTORY` must contain the two named JSONL files. `TRAINING_OUTPUT_DIRECTORY` is the base directory where all checkpoints, logs, tokenizer files, metadata, and the `adapter/` LoRA output are saved.

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# EDIT FOR THE LINUX HOST: local Qwen/Qwen3-1.7B base-model directory.
MODEL_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'models' / 'Qwen3-1.7B'
# EDIT FOR THE LINUX HOST: directory containing train_dataset_v2.jsonl and validation_dataset_v2.jsonl.
DATASET_DIRECTORY = PROJECT_ROOT / 'ticket-classification' / 'artifacts' / 'datasets'
# EDIT FOR THE LINUX HOST: base directory for checkpoints, logs, tokenizer, metadata, and LoRA adapter.
TRAINING_OUTPUT_DIRECTORY = PROJECT_ROOT / 'ticket-classification' / 'artifacts' / 'models' / 'demo09-qwen3-1.7b-ticket-classification-lora'

TRAIN_FILE = DATASET_DIRECTORY / 'train_dataset_v2.jsonl'
VALIDATION_FILE = DATASET_DIRECTORY / 'validation_dataset_v2.jsonl'
ADAPTER_OUTPUT_DIRECTORY = TRAINING_OUTPUT_DIRECTORY / 'adapter'
for required in (MODEL_DIRECTORY / 'config.json', MODEL_DIRECTORY / 'tokenizer.json', TRAIN_FILE, VALIDATION_FILE):
    if not required.is_file():
        raise FileNotFoundError(f'Missing required local artifact: {required}')
if not (MODEL_DIRECTORY / 'model.safetensors').is_file() and not (MODEL_DIRECTORY / 'model.safetensors.index.json').is_file():
    raise FileNotFoundError(f'Missing model weights in {MODEL_DIRECTORY}.')
print(f'Model directory: {MODEL_DIRECTORY}')
print(f'Dataset directory: {DATASET_DIRECTORY}')
print(f'LoRA output base directory: {TRAINING_OUTPUT_DIRECTORY}')


## CUDA GPU availability check

Run this dedicated check before model loading. The notebook stops if CUDA or CUDA BF16 is unavailable; it does not fall back to another device.

In [ ]:
DEVICE = require_cuda_bfloat16()
print(f'PyTorch version: {torch.__version__}')
print(f'Compiled CUDA version: {torch.version.cuda}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA BF16 supported: {torch.cuda.is_bf16_supported()}')
print(f'GPU count: {torch.cuda.device_count()}')
print(f'Selected GPU: {torch.cuda.get_device_name(DEVICE)}')
print(f'Selected device: {DEVICE}')


## Configuration, data validation, and training

The best checkpoint is the epoch with maximum generated validation Category+Severity Accuracy. The held-out test split is intentionally not loaded.

In [ ]:
SEED = 42
MODEL_DTYPE = torch.bfloat16
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
LEARNING_RATE, NUM_TRAIN_EPOCHS = 1e-4, 8
PER_DEVICE_TRAIN_BATCH_SIZE, PER_DEVICE_EVAL_BATCH_SIZE = 1, 1
GRADIENT_ACCUMULATION_STEPS, MAX_SEQUENCE_LENGTH = 8, 512
GENERATION_MAX_NEW_TOKENS, VALIDATION_GENERATION_BATCH_SIZE = 128, 4
ALLOWED_CATEGORIES = {'APP-01', 'AUTH-01', 'DB-01', 'DB-02', 'DEV-01', 'MAIL-01', 'MOB-01', 'NET-01', 'SEC-01', 'SRV-01'}
ALLOWED_SEVERITIES = {'P1', 'P2', 'P3', 'P4'}
set_seed(SEED)
raw_datasets = load_dataset('json', data_files={'train': str(TRAIN_FILE), 'validation': str(VALIDATION_FILE)})
train_texts = validate_split('train', raw_datasets['train'])
validation_texts = validate_split('validation', raw_datasets['validation'])
if train_texts & validation_texts:
    raise ValueError('Training and validation splits share exact ticket text.')
sft_datasets = raw_datasets.map(to_prompt_completion, remove_columns=raw_datasets['train'].column_names)
print(f"Training records: {len(sft_datasets['train'])}")
print(f"Validation records: {len(sft_datasets['validation'])}")
print(f'Model dtype: {MODEL_DTYPE}')


## Train and save the best LoRA adapter

The base model remains local and unchanged. Only LoRA adapter files and training metadata are written below the configured output base directory.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIRECTORY, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_DIRECTORY, dtype=MODEL_DTYPE, local_files_only=True).to(DEVICE)
if model.config.model_type != 'qwen3':
    raise RuntimeError(f'Expected a Qwen3 base model, got {model.config.model_type!r}.')
model.config.use_cache = False
lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none', target_modules=LORA_TARGET_MODULES)
model = get_peft_model(model, lora_config, autocast_adapter_dtype=False)
training_arguments = SFTConfig(output_dir=str(TRAINING_OUTPUT_DIRECTORY), learning_rate=LEARNING_RATE, num_train_epochs=NUM_TRAIN_EPOCHS, per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE, per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, max_length=MAX_SEQUENCE_LENGTH, eval_strategy='epoch', save_strategy='epoch', logging_strategy='steps', logging_steps=5, save_total_limit=1, load_best_model_at_end=True, metric_for_best_model='eval_joint_accuracy', greater_is_better=True, completion_only_loss=True, gradient_checkpointing=True, bf16=True, bf16_full_eval=True, dataloader_pin_memory=True, optim='adamw_torch', report_to='none', seed=SEED)
callback = ValidationGenerationMetricsCallback(model, tokenizer, raw_datasets['validation'])
trainer = SFTTrainer(model=model, args=training_arguments, train_dataset=sft_datasets['train'], eval_dataset=sft_datasets['validation'], processing_class=tokenizer, callbacks=[callback])
floating_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.is_floating_point()}
trainable_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.requires_grad}
if floating_dtypes != {MODEL_DTYPE} or trainable_dtypes != {MODEL_DTYPE}:
    raise RuntimeError(f'Expected BF16 floating and trainable parameters, got {floating_dtypes} and {trainable_dtypes}.')
trainer.model.print_trainable_parameters()
result = trainer.train()
trainer.save_model(ADAPTER_OUTPUT_DIRECTORY)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIRECTORY)
print(f'Training loss: {result.training_loss:.4f}')
print(f'Adapter saved to: {ADAPTER_OUTPUT_DIRECTORY}')


## Review loss and validation metrics

This display summarizes training and validation loss together with generated validation metrics. It does not change the restored best adapter.

In [ ]:
training_logs = [entry for entry in trainer.state.log_history if 'loss' in entry and 'epoch' in entry]
validation_logs = [entry for entry in trainer.state.log_history if 'eval_loss' in entry and 'epoch' in entry]
if not training_logs or not validation_logs or not callback.epoch_metrics:
    raise RuntimeError('Training or validation metrics are unavailable.')
fig, axis = plt.subplots(figsize=(10, 6))
axis.plot([entry['epoch'] for entry in training_logs], [entry['loss'] for entry in training_logs], label='Training loss')
axis.plot([entry['epoch'] for entry in validation_logs], [entry['eval_loss'] for entry in validation_logs], label='Validation loss')
axis.set(title='Training and Validation Loss by Epoch', xlabel='Epoch', ylabel='Loss')
axis.grid(True, linestyle='--', alpha=0.55); axis.legend(); plt.show()
lines = ['| Epoch | Category Accuracy | Severity Accuracy | Category+Severity Accuracy | Valid JSON Rate |', '| ---: | ---: | ---: | ---: | ---: |']
for row in callback.epoch_metrics:
    lines.append(f"| {row['epoch']:.0f} | {row['category_accuracy']:.1%} | {row['severity_accuracy']:.1%} | {row['joint_accuracy']:.1%} | {row['valid_json_rate']:.1%} |")
display(Markdown('\n'.join(lines)))
